<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# Hello, FABRIC: Create Your First Experiment

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook walks you through creating your very first experiment on the FABRIC testbed. By the end, you will have deployed a virtual machine on FABRIC, executed a command on it, and cleaned up your resources.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Import and initialize the FABlib library
2. View your FABRIC configuration and available testbed sites
3. Create a **slice** containing a single compute node
4. Execute a remote command on a FABRIC node
5. Delete a slice and release resources

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must** complete the environment setup:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook to create your `fabric_rc` and `ssh_config` files
2. Have your bastion host username and private key ready (provided when you joined FABRIC)

If you are using the **FABRIC JupyterHub**, most environment variables are set automatically. You still need to upload your bastion private key and set its path.

**Need help?** Visit the [FABRIC User Forum](https://learn.fabric-testbed.net/forums/) or read about [logging into FABRIC VMs](https://learn.fabric-testbed.net/knowledge-base/logging-into-fabric-vms/).

</div>

## Background: What is a Slice?

A **slice** is FABRIC's fundamental unit of resource allocation. When you create a slice, you are reserving a set of resources (virtual machines, network links, storage, GPUs, etc.) across one or more FABRIC sites. Think of a slice as your own private mini-infrastructure for running an experiment.

<img src="./figs/slice_concept.png" width="50%"><br>

In this first experiment we will create the simplest possible slice: one node on one site.

---

## Step 1: Import FABlib and Verify Configuration

Every FABRIC notebook starts by importing the **FABlib** library and creating a `FablibManager` instance. The `show_config()` call prints your current configuration so you can verify that tokens, keys, and paths are set correctly.

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

<div class="fab-warning">

**Tip:** If `show_config()` shows missing or incorrect values, go back and re-run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook.

</div>

## Step 2 (Optional): Browse Available Sites

FABRIC has 35+ sites across the US and internationally. Each site has different resource capacities. Use `list_sites()` to see what is available right now. This is useful for choosing a site with enough capacity for your experiment.

In [ ]:
# List all FABRIC sites and their available resources
fablib.list_sites();

## Step 3: Create Your Slice

Creating a slice is a three-step process:

1. **`new_slice()`** &mdash; Create an empty slice with a name
2. **`add_node()`** &mdash; Add one or more nodes (VMs) to the slice
3. **`submit()`** &mdash; Send the request to FABRIC to provision your resources

By default, `submit()` blocks until the slice is ready (typically 2-5 minutes) and displays a progress indicator.

<img src="./figs/SingleNode.png" width="20%"><br>

In [ ]:
# Step 3a: Create a new empty slice
slice = fablib.new_slice(name="MySlice")

# Step 3b: Add a single node with default settings
# (random site, 2 cores, 8 GB RAM, 10 GB disk, default Ubuntu image)
node = slice.add_node(name="Node1")

# Step 3c: Submit the slice request to FABRIC
# This will block until the slice is ready (~2-5 minutes)
slice.submit();

<div class="fab-success">

**What just happened?** FABRIC received your request, found a site with available resources, created a virtual machine, installed the operating system, configured SSH access through the bastion host, and made the node ready for your experiment.

</div>

## Step 4: Inspect Your Slice

Once the slice is active, you can inspect its attributes and list the nodes it contains.

In [ ]:
# Show slice-level information (state, expiration, project, etc.)
slice.show();

In [ ]:
# List all nodes in the slice with their details
slice.list_nodes();

## Step 5: Run a Command on Your Node

FABlib lets you execute commands on your nodes programmatically using `node.execute()`. Under the hood, this SSHs through the FABRIC bastion host into your VM and runs the command.

The method returns a tuple of `(stdout, stderr)` strings.

In [ ]:
# Execute a command on each node in the slice
for node in slice.get_nodes():
    stdout, stderr = node.execute('echo Hello, FABRIC from node `hostname -s`')

<div class="fab-success">

**Success!** If you see `Hello, FABRIC from node ...` above, your environment is correctly configured and you can create and access FABRIC resources.

</div>

## Step 6: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource &mdash; leaving slices running unnecessarily prevents other researchers from using those resources. Slices also have an expiration time; after that, they are automatically deleted.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `show_config()` shows missing values | Environment not configured | Run [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) |
| Slice stuck in `Configuring` state | Site may be busy or down | Try a different site by specifying `site='SITENAME'` in `add_node()` |
| `Permission denied` on SSH | Bastion key not configured | Check your bastion private key path in the environment config |
| `submit()` times out | Network issue or high demand | Retry with `slice.submit(wait_timeout=600)` for a longer timeout |
| `PDP Authorization check failed` | Project permissions issue | Contact your project lead or FABRIC support |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.list_sites()` | List all FABRIC sites and available resources | [list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites) |
| `fablib.new_slice(name)` | Create a new empty slice | [new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice) |
| `slice.add_node(name)` | Add a compute node to the slice | [add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.show()` | Display slice attributes | [show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show) |
| `slice.list_nodes()` | List all nodes in the slice | [list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodes) |
| `slice.get_nodes()` | Get list of node objects | [get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

Now that you have created your first slice, explore these notebooks to learn more:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Customizing Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **Listing Resources** | [list_all_resources](../sites_and_resources/list_all_resources.ipynb) | Query available capacity across all sites |
| **Running Commands** | [execute_commands](../ssh_to_nodes/execute_commands.ipynb) | Execute commands, capture output, use threads |
| **Networking** | [FABnet IPv4 (auto)](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_auto.ipynb) | Connect nodes across sites with Layer 3 networking |
| **GPUs** | [fabric_gpu](../fabric_all_gpus/fabric_gpu.ipynb) | Use NVIDIA GPUs (T4, RTX6000, A30, A40) |